# Reichman University NLP Project

Cross-request **token reuse** across LLM workloads. This notebook is fully
self-contained: every function it needs is defined below — the only imports are
the standard library plus `transformers` / `datasets` / `huggingface_hub` /
`pyarrow` / `pandas` (auto-installed by the first cell). No repo files required.

Each cell loads a corpus, applies its treatment, and measures reuse with the
accounting rules defined in the core cell (second-occurrence reuse, whole-stream
denominator, one trajectory per task, ≥500-token spans, prefix-cache-disjoint
credit). Set `HF_TOKEN` for the HuggingFace-hosted corpora (some, like WildChat,
are gated). Local-data corpora read from `./data/...`; missing data is skipped,
not faked. Run top to bottom.


## Core — tokenizer, span accounting, treatments

In [ ]:
import os, sys, re, json, glob, csv, sqlite3, subprocess
from collections import Counter, defaultdict

try:
    import pandas as pd
    from transformers import AutoTokenizer
    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem, HfApi, hf_hub_download, snapshot_download
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers", "datasets", "huggingface_hub", "pyarrow", "pandas"],
                   check=True)
    import pandas as pd
    from transformers import AutoTokenizer
    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem, HfApi, hf_hub_download, snapshot_download

# Some corpora are gated; set your token here or in the environment:
#   os.environ["HF_TOKEN"] = "hf_..."
HF_TOKEN = os.environ.get("HF_TOKEN")

MIN = 500
TOK = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B", token=HF_TOKEN)
_tc = {}
def nt(s):
    v = _tc.get(s)
    if v is None:
        v = len(TOK(s, add_special_tokens=False)["input_ids"]); _tc[s] = v
    return v

def lines_of(t):
    "Byte-faithful units: each line carries its newline; blank lines kept."
    return [l + "\n" for l in t.split("\n")]

def spans_merge(units):
    "Group consecutive units into >=MIN-token spans; a sub-MIN tail merges back."
    out, buf, n = [], [], 0
    for u in units:
        buf.append(u); n += nt(u)
        if n >= MIN:
            out.append(["".join(buf), n]); buf = []; n = 0
    if buf and out:
        out[-1][0] += "".join(buf); out[-1][1] += n
    return out

def _fnv(s):
    h = 1469598103934665603
    for c in s.encode():
        h = ((h ^ c) * 1099511628211) & ((1 << 64) - 1)
    return h

def measure_disjoint(sessions, sidecar=0):
    "Prefix-cache-disjoint accounting: prefix caching first, PIC only beyond it."
    def _spans(units, base):
        out, buf, n, start = [], [], 0, 0
        for i, u in enumerate(units):
            buf.append(u); n += nt(u)
            if n >= MIN:
                out.append([base + start, base + i + 1, "".join(buf), n]); buf = []; n = 0; start = i + 1
        if buf and out:
            out[-1][1] = base + len(units); out[-1][2] += "".join(buf); out[-1][3] += n
        return out
    prefix_seen = set(); span_seen = {}
    denom = prefix_tok = reu = w = x = 0
    for sid, sess in enumerate(sessions):
        snaps = sess["snaps"]; flat = [u for sn in snaps for u in sn]
        denom += sum(nt(u) for u in flat)
        cut, h = 0, 0
        for i, u in enumerate(flat):
            h = ((h * 1099511628211) ^ _fnv(u)) & ((1 << 64) - 1)
            if h in prefix_seen: cut = i + 1
            else: break
        prefix_tok += sum(nt(u) for u in flat[:cut])
        h = 0
        for u in flat[:2000]:
            h = ((h * 1099511628211) ^ _fnv(u)) & ((1 << 64) - 1); prefix_seen.add(h)
        base = 0
        for sn in snaps:
            for a, b, sp, n in _spans(sn, base):
                if b <= cut: continue
                k = _fnv(sp); p = span_seen.get(k)
                if p is None: span_seen[k] = sid
                elif p == sid: reu += n; w += n
                else: reu += n; x += n
            base += len(sn)
    denom += sidecar
    P = lambda a: round(100.0 * a / denom, 2) if denom else 0
    return {"prefix_caching_pct": P(prefix_tok), "pic_pct": P(reu),
            "in_session": P(w), "cross_session": P(x)}

BRACKET = re.compile(r"\[(\d+)\]")
def apply_spec(text, spec):
    "Mask/strip volatile fields (json:/yaml:/attr:/bracket_id) for treatments."
    for f in spec:
        if f.startswith("attr:"):
            text = re.sub(r'\s*' + re.escape(f[5:]) + r'="[^"]*"', "", text)
        elif f == "bracket_id":
            text = BRACKET.sub("[N]", text)
        elif f.startswith("json:"):
            k = re.escape(f[5:])
            text = re.sub(r'"' + k + r'"\s*:\s*(?:"[^"]{0,120}"|-?\d[\d.eE+-]*)',
                          '"' + f[5:] + '":"<V>"', text)
        elif f.startswith("yaml:"):
            k = re.escape(f[5:])
            text = re.sub(r"(?m)^(\s*(?:- )?)" + k + r":\s+\S.*$", r"\g<1>" + f[5:] + ": <V>", text)
    return text

ROWS, SKIPPED = [], []
def add(family, corpus, treat, prefix, pic, pic_proc, in_sess, cross_sess):
    ROWS.append({"Workload family": family, "Corpus": corpus, "Prefix caching": prefix,
                 "PIC": pic, "PIC proc.": pic_proc, "in-sess.": in_sess,
                 "cross-sess.": cross_sess, "Treatment": treat})
def stage(name):
    def deco(fn):
        try: fn(); print("OK  ", name)
        except Exception as e: SKIPPED.append((name, repr(e))); print("skip", name, "->", type(e).__name__)
    return deco

## Chat prompts — WildChat-1M, PAWS

In [ ]:
WC_DATASET, WC_SHARDS = "allenai/WildChat-1M", 14
def iter_first_turns(max_rows):
    fs = HfFileSystem(); y = 0
    for si in range(WC_SHARDS):
        path = f"datasets/{WC_DATASET}/data/train-{si:05d}-of-{WC_SHARDS:05d}.parquet"
        with fs.open(path) as f:
            pf = pq.ParquetFile(f)
            for rg in range(pf.num_row_groups):
                tbl = pf.read_row_group(rg, columns=["conversation"])
                for c in tbl.column("conversation").to_pylist():
                    text = ""
                    for turn in c or []:
                        if turn.get("role") == "user":
                            text = turn.get("content") or ""; break
                    if not text and c:
                        text = c[0].get("content", "")
                    if not text:
                        continue
                    yield text; y += 1
                    if y >= max_rows:
                        return

@stage("WildChat-1M (40K)")
def _():
    texts = list(iter_first_turns(40_000))
    d = measure_disjoint([{"task": i, "snaps": [lines_of(t)]} for i, t in enumerate(texts)])
    add("Chat prompts", "WildChat-1M (40K)", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

@stage("PAWS (40K)")
def _():
    from datasets import load_dataset
    ds = load_dataset("google-research-datasets/paws", "labeled_final",
                      split="train", streaming=True)
    texts, n = [], 0
    for r in ds:
        for k in ("sentence1", "sentence2"):
            if r.get(k):
                texts.append(r[k]); n += 1
        if n >= 40_000:
            break
    d = measure_disjoint([{"task": i, "snaps": [lines_of(t)]} for i, t in enumerate(texts)])
    add("Paraphrase traffic", "PAWS (40K)", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

## Coding agents — SWE-smith, OpenHands, CC-Bench, SWE-agent

Whole-prompt reuse under the five rules: the system prompt is credited to prefix
caching, PIC spans are cut from each newly appended message (blake2b span keys,
line+newline token counts). One trajectory per task.

In [ ]:
import hashlib
def h64(s):
    return int.from_bytes(hashlib.blake2b(s.encode("utf-8", "replace"), digest_size=8).digest(), "big")

def m_pack_tokens(cum, target, min_tail=0):
    n = len(cum) - 1; spans = []; start = 0
    while start < n:
        i = start
        while i < n and cum[i + 1] - cum[start] < target: i += 1
        end = min(i + 1, n); spans.append((start, end)); start = end
    if min_tail and len(spans) > 1:
        a, b = spans[-1]
        if cum[b] - cum[a] < min_tail:
            spans[-2] = (spans[-2][0], b); spans.pop()
    return spans

HEAD = re.compile(r"result of running `cat -n` on (?:a snippet of )?([^\n:]+):")
SANDBOX = re.compile(r"^(?:/testbed/|/workspace/[^/]+/|/workspace/|/app/projects/|/app/)")
CORPORA = {
    "swesmith":  {"repo": "SWE-bench/SWE-smith-trajectories",
                  "files": [f"data/train-{i:05d}-of-00008.parquet" for i in range(8)],
                  "cols": ["messages", "instance_id"], "format": "cat_n"},
    "openhands": {"repo": "nebius/SWE-rebench-openhands-trajectories",
                  "files": ["trajectories.parquet"],
                  "cols": ["trajectory", "instance_id"], "format": "cat_n"},
    "sweagent":  {"repo": "nebius/SWE-agent-trajectories",
                  "files": [f"data/train-{i:05d}-of-00012.parquet" for i in range(12)],
                  "cols": ["trajectory", "instance_id"], "format": "swe_agent"},
    "ccbench":   {"repo": "zai-org/CC-Bench-trajectories", "files": ["train.parquet"],
                  "cols": ["trajectory", "task_id"], "format": "claude_code"},
}
def repo_of(instance_id):
    head = str(instance_id).split("__", 1)
    if len(head) < 2: return str(instance_id)
    base = head[1].split(".", 1)[0]; base = re.sub(r"-\d+$", "", base)
    return f"{head[0]}/{base}"
def _text_of(content):
    if isinstance(content, str): return content
    if isinstance(content, list):
        return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
    return ""

def iter_messages(corpus, max_trajs=None, one_per_task=False):
    "Yield {traj_id, repo, messages:[(role,text)]}, role in {system,input,decode}."
    cfg = CORPORA[corpus]; msg_col, id_col = cfg["cols"]; fmt = cfg["format"]
    seen_ids = set(); n = 0
    for rf in cfg["files"]:
        fs = HfFileSystem(skip_instance_cache=True)
        with fs.open(f"datasets/{cfg['repo']}/{rf}") as f:
            pf = pq.ParquetFile(f)
            for rg in range(pf.num_row_groups):
                t = pf.read_row_group(rg, columns=cfg["cols"])
                for raw, iid in zip(t.column(msg_col).to_pylist(), t.column(id_col).to_pylist()):
                    if one_per_task:
                        if str(iid) in seen_ids: continue
                        seen_ids.add(str(iid))
                    msgs = json.loads(raw) if isinstance(raw, str) else raw
                    out = []
                    if fmt == "claude_code":
                        for ev in msgs or []:
                            m = ev.get("message") or {}; role = m.get("role"); c = m.get("content")
                            if isinstance(c, str):
                                out.append(("decode" if role == "assistant" else "input", c))
                            elif isinstance(c, list):
                                for b in c:
                                    if not isinstance(b, dict): continue
                                    if b.get("type") == "text":
                                        out.append(("decode" if role == "assistant" else "input", b.get("text") or ""))
                                    elif b.get("type") == "tool_result":
                                        out.append(("input", _text_of(b.get("content"))))
                                    elif b.get("type") == "tool_use":
                                        out.append(("decode", json.dumps(b.get("input") or {})))
                    elif fmt == "swe_agent":
                        for m in msgs or []:
                            if m.get("system_prompt"): out.append(("system", m["system_prompt"]))
                            if m.get("text"):
                                out.append(("decode" if m.get("role") == "ai" else "input", m["text"]))
                    else:
                        for m in msgs or []:
                            c = m.get("content") or ""; r = m.get("role")
                            out.append(("system" if r == "system" else "decode" if r == "assistant" else "input", c))
                    out = [(r, x) for r, x in out if x]
                    if not out: continue
                    yield {"traj_id": n,
                           "repo": (f"task-{iid}" if fmt == "claude_code" else repo_of(iid or "")),
                           "messages": out}
                    n += 1
                    if max_trajs and n >= max_trajs: return

def coding_run(corpus, max_trajs=1000):
    tcache = {}
    def ntok(s):
        v = tcache.get(s)
        if v is None:
            v = len(TOK(s + "\n", add_special_tokens=False)["input_ids"])
            if len(tcache) < 4_000_000: tcache[s] = v
        return v
    seen = {}; repos = {}
    total = reused = within = cross_same = cross_diff = sys_prefix = 0
    for tr in iter_messages(corpus, max_trajs, one_per_task=True):
        rid = repos.setdefault(tr["repo"], len(repos))
        for role, text in tr["messages"]:
            if role == "decode": continue
            if role == "system":
                st = sum(ntok(ln) for ln in text.split("\n")); total += st; sys_prefix += st; continue
            new = {}; lines = text.split("\n"); cum = [0]
            for ln in lines: cum.append(cum[-1] + ntok(ln))
            if not cum[-1]: continue
            for a, b in m_pack_tokens(cum, MIN, min_tail=MIN):
                ntk = cum[b] - cum[a]; total += ntk
                if ntk < MIN: continue
                key = h64("\n".join(lines[a:b])); prev = seen.get(key)
                if prev is None:
                    new.setdefault(key, ntk)
                else:
                    reused += ntk
                    if prev[0] == tr["traj_id"]: within += ntk
                    elif prev[1] == rid: cross_same += ntk
                    else: cross_diff += ntk
            for k in new: seen.setdefault(k, (tr["traj_id"], rid))
    p = lambda a: round(100.0 * a / total, 2) if total else 0.0
    return {"prefix": p(sys_prefix), "pic": p(reused),
            "in": p(within), "cross": p(cross_same + cross_diff)}

for corpus, family, label in [("swesmith", "Coding agents", "SWE-smith (1K)"),
                              ("openhands", "Coding agents", "OpenHands (1K)"),
                              ("ccbench", "Coding agents", "CC-Bench (74)"),
                              ("sweagent", "Coding agents", "SWE-agent (1K)")]:
    @stage(label)
    def _(corpus=corpus, family=family, label=label):
        r = coding_run(corpus, 1000)
        add(family, label, "none", r["prefix"], r["pic"], r["pic"], r["in"], r["cross"])

## Web agents — NNetNav-WA, NNetNav-Live, Mind2Web

WebArena and NNetNav-Live share a fixed instruction head per step, so they use
the head-counted-once denominator; Mind2Web (raw HTML, no repeated head) uses the
per-snapshot disjoint measure. Treatment `replace`: volatile handles → stable
content-derived handles.

In [ ]:
BS = 16
OBS_HEAD = re.compile(r"OBSERVATION:\s*\n", re.S)
IDATTR = re.compile(r'\s*backend_node_id="[^"]*"')
def _fnv16(s):
    h = 1469598103934665603
    for c in s.encode(): h = ((h ^ c) * 1099511628211) & ((1 << 64) - 1)
    return h

def replace_handles(snapshot):                 # a11y tree: [123] -> stable content handle
    occ, out = Counter(), []
    for line in snapshot.split("\n"):
        cleaned = BRACKET.sub("[.]", line); occ[cleaned] += 1; o = occ[cleaned] - 1; i = [0]
        def sub(m):
            aid = _fnv16(f"{cleaned}|{o}|{i[0]}"); i[0] += 1; return f"[a{aid:016x}]"
        out.append(BRACKET.sub(sub, line))
    return "\n".join(out)

def replace_ids(html):                          # Mind2Web: backend_node_id -> content handle
    occ = Counter()
    def sub(m):
        seg = m.group(0); cleaned = IDATTR.sub("", seg); occ[cleaned] += 1
        return re.sub(r'backend_node_id="[^"]*"',
                      f'backend_node_id="a{_fnv16(cleaned + "|" + str(occ[cleaned]-1)):016x}"', seg)
    return re.sub(r"<[^>]*backend_node_id=\"[^\"]*\"[^>]*>", sub, html)

def web_run(repo, treat, max_tasks=40, max_lines=4000):
    p = hf_hub_download(repo, "train.jsonl", repo_type="dataset")
    by = defaultdict(list)
    for i, line in enumerate(open(p)):
        if i >= max_lines: break
        r = json.loads(line)
        if r.get("prompt"): by[r.get("id")].append(r["prompt"])
    tasks = [v for v in by.values() if len(v) >= 2][:max_tasks]
    prefix_cache, seen = set(), {}
    denom = prefix_served = pic_in = pic_cross = 0
    for sid, prompts in enumerate(tasks):
        for step, prompt in enumerate(prompts):
            m = OBS_HEAD.search(prompt)
            head = prompt[:m.end()] if m else prompt
            obs = prompt[m.end():] if m else ""
            if treat and obs: obs = treat(obs)
            ids = TOK(head + obs, add_special_tokens=False)["input_ids"]
            head_n = len(TOK(head, add_special_tokens=False)["input_ids"])
            obs_n = len(ids) - head_n
            h = 0; chain = []; hits = 0; counting = True
            for i in range(len(ids) // BS):
                h = hash((h, tuple(ids[i*BS:(i+1)*BS]))); chain.append(h)
                if counting:
                    if h in prefix_cache: hits += 1
                    else: counting = False
            for h2 in chain: prefix_cache.add(h2)
            served = hits * BS
            if step == 0:
                denom += head_n + obs_n
                prefix_served += min(served, head_n) + max(0, served - head_n)
            else:
                denom += obs_n; prefix_served += max(0, served - head_n)
            skip = max(0, served - head_n); kept, acc = [], 0
            for l in [x for x in obs.split("\n") if x.strip()]:
                t = nt(l)
                if acc + t <= skip: acc += t; continue
                kept.append(l)
            for sp, tn in spans_merge(kept):
                k = hash(sp); pp = seen.get(k)
                if pp is None: seen[k] = sid
                elif pp == sid: pic_in += tn
                else: pic_cross += tn
    P = lambda a: round(100.0 * a / denom, 2) if denom else 0
    return {"prefix_caching_pct": P(prefix_served), "pic_pct": P(pic_in + pic_cross),
            "pic_in_sess_pct": P(pic_in), "pic_cross_sess_pct": P(pic_cross)}

for repo, family, corpus in [("stanfordnlp/nnetnav-wa", "Web pages (a11y)", "NNetNav-WA (WebArena)"),
                             ("stanfordnlp/nnetnav-live", "Web pages (live)", "NNetNav-Live")]:
    @stage(corpus)
    def _(repo=repo, family=family, corpus=corpus):
        v = web_run(repo, None); t = web_run(repo, replace_handles)
        add(family, corpus, "replace", t["prefix_caching_pct"], v["pic_pct"],
            t["pic_pct"], t["pic_in_sess_pct"], t["pic_cross_sess_pct"])

def sessions_mind2web(max_tasks, tries=6):
    from datasets import load_dataset
    out, consumed = [], 0
    for attempt in range(tries):
        try:
            ds = load_dataset("osunlp/Mind2Web", split="train", streaming=True)
            for i, ex in enumerate(ds):
                if i < consumed: continue
                consumed = i + 1
                snaps = [a.get("cleaned_html") or "" for a in (ex.get("actions") or [])]
                snaps = [s for s in snaps if s]
                if len(snaps) >= 2: out.append(snaps)
                if len(out) >= max_tasks: return out
            return out
        except Exception:
            if attempt == tries - 1: raise
    return out

@stage("Mind2Web")
def _():
    m2 = sessions_mind2web(25)
    def m(xform):
        return measure_disjoint([{"task": i, "snaps": [lines_of(xform(sn) if xform else sn) for sn in snaps]}
                                 for i, snaps in enumerate(m2)])
    v, t = m(None), m(replace_ids)
    add("Web pages (HTML)", "Mind2Web", "replace",
        v["prefix_caching_pct"], v["pic_pct"], t["pic_pct"], t["in_session"], t["cross_session"])

## Operational telemetry — ITBench SRE / K8s / alerts, BGL syslog

`relocate` moves volatile fields into a sidecar (still in the denominator) so the
stable body forms reusable spans (SRE, BGL); K8s and alerts are untreated. BGL
reads local `data/loghub/BGL_200k.log`.

In [ ]:
def _snapshot_dl(repo, patterns):
    try:
        return snapshot_download(repo, repo_type="dataset", allow_patterns=patterns, local_files_only=True)
    except Exception:
        return snapshot_download(repo, repo_type="dataset", allow_patterns=patterns)

@stage("ITBench SRE")
def _():
    TASK = re.compile(r"(Scenario-\d+)"); REPO = "ibm-research/ITBench-Trajectories"
    files = sorted(s.rfilename for s in HfApi().dataset_info(REPO).siblings
                   if s.rfilename.endswith("session.jsonl"))
    sess = []
    for rf in files:
        p = hf_hub_download(REPO, rf, repo_type="dataset")
        m = TASK.search(rf); task = m.group(1) if m else rf
        raw_s, rel_s, side = [], [], 0
        for line in open(p):
            try: r = json.loads(line)
            except ValueError: continue
            pl = r.get("payload") or {}
            if pl.get("type") != "function_call_output": continue
            o = pl.get("output") or ""; raw = o
            try:
                inner = json.loads(o)
                if isinstance(inner, dict) and isinstance(inner.get("output"), str) and inner.get("metadata") is not None:
                    o = inner["output"]; side += nt(json.dumps(inner["metadata"], ensure_ascii=False))
                else:
                    o = json.dumps(inner, ensure_ascii=False); raw = o
            except (ValueError, TypeError): pass
            if len(o) >= 80: rel_s.append(lines_of(o)); raw_s.append(lines_of(raw))
        if len(rel_s) >= 2: sess.append({"task": task, "raw": raw_s, "rel": rel_s, "side": side})
        if len(sess) >= 40: break
    seen = set(); dd = [s for s in sess if not (s["task"] in seen or seen.add(s["task"]))]
    v = measure_disjoint([{"task": s["task"], "snaps": s["raw"]} for s in dd])
    t = measure_disjoint([{"task": s["task"], "snaps": s["rel"]} for s in dd], sidecar=sum(s["side"] for s in dd))
    add("Op. telemetry", "ITBench SRE", "relocate", v["prefix_caching_pct"], v["pic_pct"],
        t["pic_pct"], t["in_session"], t["cross_session"])

@stage("ITBench infra compliance (K8s)")
def _():
    root = _snapshot_dl("ibm-research/ITBench-Lite", ["snapshots/ciso/**"]); s = []
    for scen in sorted(glob.glob(os.path.join(root, "snapshots/ciso/*/*"))):
        snaps = [open(p).read() for p in sorted(glob.glob(os.path.join(scen, "static-resources*/**/*.yaml"), recursive=True))]
        snaps = [x for x in snaps if len(x) >= 80]
        if len(snaps) >= 2: s.append(snaps)
        if len(s) >= 200: break
    d = measure_disjoint([{"task": i, "snaps": [lines_of(sn) for sn in snaps]} for i, snaps in enumerate(s)])
    add("Op. telemetry", "ITBench infra compliance (K8s)", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

@stage("ITBench alerts")
def _():
    root = _snapshot_dl("ArtificialAnalysis/ITBench-AA", ["sre/**"]); s = []
    for scen in sorted(glob.glob(os.path.join(root, "sre/Scenario-*"))):
        snaps = [open(p).read() for p in sorted(glob.glob(os.path.join(scen, "alerts/alerts_at_*.json")))]
        snaps = [x for x in snaps if len(x) >= 80]
        if len(snaps) >= 2: s.append(snaps)
        if len(s) >= 40: break
    d = measure_disjoint([{"task": i, "snaps": [lines_of(sn) for sn in snaps]} for i, snaps in enumerate(s)])
    add("Op. telemetry", "ITBench alerts", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

@stage("BGL syslog (LogHub)")
def _():
    POS = (1, 4)
    lines = [l.rstrip("\n") for l in open("data/loghub/BGL_200k.log") if l.strip()]
    per = len(lines) // 20; raw, rel, side = [], [], 0
    for s in range(20):
        chunk = lines[s*per:(s+1)*per]; rs, ts = [], []
        for i in range(0, len(chunk), 500):
            ru, tu = [], []
            for line in chunk[i:i+500]:
                toks = line.split(); ru.append(line + "\n")
                if len(toks) >= 8:
                    side += nt(" ".join(toks[j] for j in POS if j < len(toks)))
                    tu.append(" ".join(t for j, t in enumerate(toks) if j not in POS) + "\n")
                else:
                    tu.append(line + "\n")
            rs.append(ru); ts.append(tu)
        raw.append({"task": s, "snaps": rs}); rel.append({"task": s, "snaps": ts})
    v = measure_disjoint(raw); t = measure_disjoint(rel, sidecar=side)
    add("Op. telemetry", "BGL syslog (LogHub)", "relocate",
        v["prefix_caching_pct"], v["pic_pct"], t["pic_pct"], t["in_session"], t["cross_session"])

## Stateful API responses — AppWorld, tau2 (airline / retail / telecom)

AppWorld uses `mask` (blank a volatile JSON field, diagnostic); tau2 is untreated.
Both read local `data/...`.

In [ ]:
@stage("AppWorld")
def _():
    MSG = re.compile(r"llm\.input_messages\.(\d+)\.message\.role")
    best = {}
    for line in open("data/appworld/hf_traces/halo_gemini3flash_traces.jsonl"):
        r = json.loads(line); a = r.get("attributes") or {}
        if a.get("openinference.span.kind") != "LLM": continue
        idxs = [int(m.group(1)) for k in a for m in [MSG.fullmatch(k)] if m]
        if idxs:
            tid = r["trace_id"]
            if tid not in best or len(idxs) > best[tid][0]: best[tid] = (len(idxs), a, max(idxs))
    aw = []
    for tid, (n, a, mx) in best.items():
        task, snaps = None, []
        for i in range(mx + 1):
            role = a.get(f"llm.input_messages.{i}.message.role")
            c = a.get(f"llm.input_messages.{i}.message.content") or ""
            if role == "user":
                m = re.search(r"# Real Task Instruction\n(.*?)(?:\n\n|$)", c, re.S)
                if m: task = m.group(1).strip()
            elif role == "tool" and isinstance(c, str) and len(c) >= 80: snaps.append(c)
        if len(snaps) >= 2 and task: aw.append({"task": task, "snaps": snaps})
    seen = set(); aw = [x for x in aw if not (x["task"] in seen or seen.add(x["task"]))]
    def m(spec):
        return measure_disjoint([{"task": x["task"],
            "snaps": [lines_of(apply_spec(sn, spec) if spec else sn) for sn in x["snaps"]]} for x in aw])
    v, t = m(None), m(["json:release_date"])
    add("Stateful API", "AppWorld", "mask", v["prefix_caching_pct"], v["pic_pct"],
        t["pic_pct"], t["in_session"], t["cross_session"])

TAU2 = re.compile(r"_(airline|retail|telecom)_")
for domain in ("airline", "retail", "telecom"):
    @stage(f"tau2 {domain}")
    def _(domain=domain):
        fs = [f for f in sorted(glob.glob("data/tau2/final/*.json"))
              if (m := TAU2.search(os.path.basename(f))) and m.group(1) == domain]
        sess = []
        for f in fs:
            for s in (json.load(open(f)).get("simulations") or []):
                snaps = [c for msg in (s.get("messages") or [])
                         if msg.get("role") == "tool" and not msg.get("error")
                         and isinstance(c := msg.get("content"), str) and len(c) >= 80]
                if len(snaps) >= 2: sess.append({"task": s.get("task_id"), "snaps": snaps})
        sess.sort(key=lambda s: str(s["task"]))
        seen = set(); dd = [s for s in sess if not (s["task"] in seen or seen.add(s["task"]))]
        d = measure_disjoint([{"task": s["task"], "snaps": [lines_of(x) for x in s["snaps"]]} for s in dd])
        add("Stateful API", f"tau2 {domain}", "none",
            d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

## Retrieved documents — MT-RAG, MultiDoc2Dial, tau-Knowledge

tau-Knowledge uses `align`: pack the KB into fixed ≥500-token buckets so a
document reused across tasks lands on the same span grid. All read local `data/...`.

In [ ]:
@stage("MT-RAG ibmcloud")
def _():
    convs = json.load(open("data/mtrag/conversations/conversations_human.json"))
    bycol = defaultdict(list)
    for i, c in enumerate(convs):
        snaps = []
        for m in c.get("messages") or []:
            ctxs = m.get("contexts") or []
            units = [p.get("text") or "" for p in ctxs if (p.get("text") or "").strip()]
            if units: snaps.append(units)
        if snaps:
            bycol[(c.get("domain", ""))].append({"task": f"{c.get('domain','')}/{i}", "snaps": snaps})
    col = next(k for k in bycol if "ibmcloud" in k)
    d = measure_disjoint(bycol[col])
    add("Retrieved docs", "MT-RAG ibmcloud", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

@stage("MultiDoc2Dial")
def _():
    docs = json.load(open("data/multidoc2dial/multidoc2dial_doc.json"))["doc_data"]
    doc_text = {did: d["doc_text"] for dd in docs.values() for did, d in dd.items()}
    out = []
    for split in ("train", "validation"):
        data = json.load(open(f"data/multidoc2dial/multidoc2dial_dial_{split}.json"))["dial_data"]
        for dials in data.values():
            for dial in dials:
                snaps = []
                for t in dial["turns"]:
                    if t.get("role") != "user": continue
                    seen = set()
                    for r in (t.get("references") or []):
                        did = r.get("doc_id")
                        if did and did not in seen and did in doc_text:
                            seen.add(did); snaps.append(lines_of(doc_text[did]))
                    if snaps: pass
                if snaps: out.append({"task": dial["dial_id"], "snaps": snaps})
    seen = set(); out = [x for x in out if not (x["task"] in seen or seen.add(x["task"]))]
    d = measure_disjoint(out)
    add("Retrieved docs", "MultiDoc2Dial", "none",
        d["prefix_caching_pct"], d["pic_pct"], d["pic_pct"], d["in_session"], d["cross_session"])

@stage("tau-Knowledge")
def _():
    docs = {}
    for f in glob.glob("data/tauknowledge/banking_knowledge/documents/*.json"):
        j = json.load(open(f)); docs[j["id"]] = j.get("title", "") + "\n" + (j.get("content") or "")
    tasks = json.load(open("data/tauknowledge/banking_knowledge/tasks.json"))
    if isinstance(tasks, dict): tasks = tasks.get("tasks") or list(tasks.values())
    sel = [{"task": t.get("id"), "req": r} for t in tasks
           if (r := [d for d in sorted(t.get("required_documents") or []) if d in docs])]
    concat = measure_disjoint(
        [{"task": t["task"], "snaps": [[u for d in t["req"] for u in lines_of(docs[d])]]} for t in sel])
    buckets, buf, n = [], [], 0
    for d in sorted(docs):
        buf.append(d); n += nt(docs[d])
        if n >= MIN: buckets.append(buf); buf = []; n = 0
    if buf: buckets[-1].extend(buf) if buckets else buckets.append(buf)
    owner = {d: i for i, b in enumerate(buckets) for d in b}
    aligned = measure_disjoint(
        [{"task": t["task"], "snaps": [[u for d in buckets[i] for u in lines_of(docs[d])]
                                       for i in sorted({owner[d] for d in t["req"]})]} for t in sel])
    add("Retrieved docs", "tau-Knowledge", "align",
        concat["prefix_caching_pct"], concat["pic_pct"], aligned["pic_pct"],
        aligned["in_session"], aligned["cross_session"])

## Database schemas — Spider 2.0, BIRD, LiveSQLBench, BIRD-INTERACT

Schema DDL packed into ≥500-token spans reused across questions on the same
database. Prefix caching reported schema-first / schema-after-context (a
16-token block chain). Spider/LiveSQL/BIRD-INTERACT read local `data/...`; BIRD
pulls its questions + schemas from HuggingFace.

In [ ]:
BS_SQL = 16
def sql_nl_spans(units):                # units joined by newline (schema DDL blocks)
    out, buf, n = [], [], 0
    for u in units:
        buf.append(u); n += nt(u)
        if n >= MIN: out.append(["\n".join(buf), n]); buf = []; n = 0
    if buf and out:
        out[-1][0] += "\n" + "\n".join(buf); out[-1][1] += n
    return out
def sql_prefix(prompts, schema_first):  # vLLM-style 16-token rolling block chain
    cache = set(); reu = tot = 0
    for p in prompts:
        schema = "\n".join(p["units"])
        text = (schema + "\n" + p["head"]) if schema_first else (p["head"] + "\n" + schema)
        ids = TOK(text, add_special_tokens=False)["input_ids"]; h = 0
        for i in range(len(ids) // BS_SQL):
            h = hash((h, tuple(ids[i*BS_SQL:(i+1)*BS_SQL]))); tot += BS_SQL
            if h in cache: reu += BS_SQL
            else: cache.add(h)
    return round(100.0 * reu / tot, 2) if tot else 0.0
def sql_pic(prompts, nl=True):
    seen = {}; reu = tot = w = x = 0
    span = sql_nl_spans if nl else spans_merge
    for p in prompts:
        tot += nt(p["head"]) + sum(nt(u) for u in p["units"])
        for sp, n in span(p["units"]):
            k = hash(sp); prev = seen.get(k)
            if prev is None: seen[k] = p["task"]
            elif prev == p["task"]: reu += n; w += n
            else: reu += n; x += n
    P = lambda a: round(100.0 * a / tot, 2) if tot else 0
    return {"total": P(reu), "same": P(w), "cross": P(x)}

@stage("Spider 2.0")
def _():
    ROOT = "data/spider2"; ALIAS = {"sqlite-sakila": "SQLITE_SAKILA", "Db-IMDB": "DB_IMDB"}
    SHARD = re.compile(r"\d{8}$"); csv.field_size_limit(10 ** 8)
    def schema_units(db):
        hits = glob.glob(os.path.join(ROOT, "databases", "*", ALIAS.get(db, db)))
        if not hits: return None
        units, seen_shard = [], set()
        for ddl in sorted(glob.glob(os.path.join(hits[0], "**", "DDL.csv"), recursive=True)):
            for r in sorted(csv.DictReader(open(ddl)), key=lambda r: r["table_name"]):
                base = SHARD.sub("<DATE>", r["table_name"])
                if base != r["table_name"]:
                    if base in seen_shard: continue
                    seen_shard.add(base)
                units.append(r.get("DDL") or r.get("ddl") or "")
        return [u for u in units if u] or None
    prompts = []
    for q in (json.loads(l) for l in open(os.path.join(ROOT, "spider2-lite.jsonl"))):
        u = schema_units(q["db"])
        if not u: continue
        doc = ""
        if q.get("external_knowledge"):
            p = os.path.join(ROOT, "documents", q["external_knowledge"])
            if os.path.exists(p): doc = open(p).read()
        prompts.append({"task": q["db"], "units": u,
                        "head": "You are a SQL agent.\nTask " + q["instance_id"]
                        + "\nQuestion: " + q["question"] + "\n" + doc})
    pic = sql_pic(prompts, nl=True)
    add("Database schemas", "Spider 2.0", "none",
        f"{sql_prefix(prompts, True)}/{sql_prefix(prompts, False)}",
        pic["total"], pic["total"], pic["same"], pic["cross"])

@stage("BIRD")
def _():
    DBS = ["california_schools", "card_games", "codebase_community", "debit_card_specializing",
           "european_football_2", "financial", "formula_1", "student_club", "superhero",
           "thrombosis_prediction", "toxicology"]
    path = "data/bird/schemas.json"
    if os.path.exists(path):
        schemas = json.load(open(path))
    else:
        os.makedirs(os.path.dirname(path), exist_ok=True); schemas = {}
        for db in DBS:
            p = hf_hub_download("target-benchmark/bird-corpus-validation",
                                f"validation_database/{db}/{db}.sqlite", repo_type="dataset")
            con = sqlite3.connect(p)
            schemas[db] = [r[0] for r in con.execute(
                "SELECT sql FROM sqlite_master WHERE sql IS NOT NULL AND type IN ('table','view') ORDER BY name")]
            con.close()
        json.dump(schemas, open(path, "w"), indent=1)
    p = hf_hub_download("birdsql/bird_sql_dev_20251106",
                        "data/dev_20251106-00000-of-00001.json", repo_type="dataset")
    prompts = []
    for q in json.load(open(p)):
        u = schemas.get(q["db_id"])
        if u:
            prompts.append({"task": q["db_id"], "units": u,
                            "head": "You are a SQL analyst.\nQuestion " + str(q["question_id"])
                            + ": " + q["question"] + "\nEvidence: " + (q.get("evidence") or "")})
    pic = sql_pic(prompts, nl=False)   # BIRD joins DDL blocks without a separator
    add("Database schemas", "BIRD", "none",
        f"{sql_prefix(prompts, True)}/{sql_prefix(prompts, False)}",
        pic["total"], pic["total"], pic["same"], pic["cross"])

def db_context_units(root, db):
    d = os.path.join(root, db); units = []
    p = os.path.join(d, db + "_schema.txt")
    if os.path.exists(p): units += [b for b in open(p).read().split("\n\n") if b.strip()]
    p = os.path.join(d, db + "_column_meaning_base.json")
    if os.path.exists(p): units += [f'{k}: {v}' for k, v in sorted(json.load(open(p)).items())]
    p = os.path.join(d, db + "_kb.jsonl")
    if os.path.exists(p): units += [l.strip() for l in open(p) if l.strip()]
    return units

@stage("LiveSQLBench")
def _():
    root = "data/livesqlbench/large-v1"
    tasks = [json.loads(l) for l in open(os.path.join(root, "livesqlbench_large_v1_data.jsonl"))]
    cache, prompts = {}, []
    for t in tasks:
        db = t["selected_database"]
        if db not in cache: cache[db] = db_context_units(root, db)
        prompts.append({"task": t["instance_id"], "units": cache[db],
                        "head": "Task " + t["instance_id"] + "\nQuestion: " + t["query"]})
    pic = sql_pic(prompts, nl=True)
    add("Database schemas", "LiveSQLBench", "none",
        f"{sql_prefix(prompts, True)}/{sql_prefix(prompts, False)}",
        pic["total"], pic["total"], pic["same"], pic["cross"])

@stage("BIRD-INTERACT")
def _():
    f = glob.glob("data/birdinteract/lite/**/bird_interact_data.jsonl", recursive=True)
    if not f: raise FileNotFoundError("data/birdinteract/lite")
    tasks = [json.loads(l) for l in open(f[0])]; base = os.path.dirname(f[0])
    cache, prompts = {}, []
    for t in tasks:
        db = t["selected_database"]
        if db not in cache: cache[db] = db_context_units(base, db)
        turns = [("t1", t.get("amb_user_query") or t["query"])]
        fu = (t.get("follow_up") or {}).get("query")
        if fu: turns.append(("t2", fu))
        for tag, q in turns:
            prompts.append({"task": t["instance_id"], "units": cache[db],
                            "head": f"Task {t['instance_id']} {tag}\nQuestion: {q}"})
    pic = sql_pic(prompts, nl=True)
    add("Conversational SQL", "BIRD-INTERACT", "none",
        f"{sql_prefix(prompts, True)}/{sql_prefix(prompts, False)}",
        pic["total"], pic["total"], pic["same"], pic["cross"])

## Results table

In [ ]:
PAPER = {  # published PIC-processed %, for a reproduction check only
    "WildChat-1M (40K)": 3.28, "PAWS (40K)": 0.00, "SWE-smith (1K)": 17.06,
    "OpenHands (1K)": 2.69, "CC-Bench (74)": 16.43, "SWE-agent (1K)": 19.27,
    "NNetNav-WA (WebArena)": 10.03, "Mind2Web": 42.94, "NNetNav-Live": 14.79,
    "ITBench SRE": 7.71, "ITBench infra compliance (K8s)": 26.73, "ITBench alerts": 16.93,
    "BGL syslog (LogHub)": 8.49, "AppWorld": 0.40, "tau2 airline": 1.31,
    "tau2 retail": 23.08, "tau2 telecom": 3.73, "MT-RAG ibmcloud": 13.76,
    "MultiDoc2Dial": 39.97, "tau-Knowledge": 47.74, "Spider 2.0": 86.95,
    "BIRD": 78.54, "LiveSQLBench": 95.86, "BIRD-INTERACT": 97.87,
}
df = pd.DataFrame(ROWS)
if not df.empty:
    df["Paper PIC proc."] = df["Corpus"].map(PAPER)
    df["Δ"] = (df["PIC proc."] - df["Paper PIC proc."]).round(2)

def bold_family_best(col):
    best = df.groupby("Workload family")["PIC proc."].transform("max")
    return ["font-weight: bold" if v == b else "" for v, b in zip(col, best)]

print(f"produced {len(ROWS)} / 24 rows; skipped {len(SKIPPED)} (missing data/token)")
df.style.apply(bold_family_best, subset=["PIC proc."]).format(precision=2).hide(axis="index")

Every value above was produced by the code in this notebook. `PIC proc.` is
reuse after the corpus's treatment; `Δ` is its gap to the published figure (0.00
= exact reproduction). PIC savings are **in addition** to prefix caching. Rows in
`SKIPPED` had no local data or HF access in this environment.